In [32]:
import pandas as pd
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration

In [33]:
train_data = pd.read_csv("samsum-train.csv")
val_data = pd.read_csv("samsum-validation.csv")

In [34]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [35]:
train_data.sample(5) #html tags, special chars, etc...

,id,dialogue,summary
942,13730026,"Justin: look, why dont you let me take you out...",Justin will pick Cecilia up at noon today and ...
8915,13730861-1,"George: Goodness, finally home!\r\nGeorge: the...",George is home after a long journey. He and Su...
4117,13727680,Suzanne: I'm rewatching friends\r\nRosanne: ha...,Suzanne and Rosanne noticed they use the same ...
11314,13728091,Linda: Babes... you were looking gorgeous last...,Linda loves Sarah's lipstick. Sarah doesn't ge...
13756,13681526,Peter: have you seen debate yesterday?\r\nAshl...,Peter and Ashley both watched a political deba...


In [36]:
print(train_data.shape)
print(val_data.shape)

(14732, 3)
(818, 3)


In [37]:
# random sampling
train_data = train_data.sample(n=4000, random_state=42).reset_index(drop=True)
val_data = val_data.sample(n=600, random_state=42).reset_index(drop=True)

In [38]:
print(train_data.shape)
print(val_data.shape)

(4000, 3)
(600, 3)


### Data Preprocessing

In [39]:
import re

def clean_data(text):
    text = re.sub(r"[\r\n]+", " ", text) # remove lines
    text = re.sub(r"\s+", " ", text) # spaces
    text = re.sub(r"<.*?>", " ", text) # html tags
    text = text.strip().lower() # remove initial spaces

    return text

In [40]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)

### Tokenize

In [41]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

In [42]:
# raw data => tokenized inputs for fine-tuning

def tokenize(data):
    inputs = tokenizer(data["dialogue"], padding="max_length", max_length=512, truncation=True)
    targets = tokenizer(data["summary"], padding="max_length", max_length=150, truncation=True)

    inputs["labels"] = targets["input_ids"] # token ids => add to input as labels
    return inputs

In [43]:
train_dataset = train_data.apply(tokenize, axis=1).tolist()
val_dataset = val_data.apply(tokenize, axis=1).tolist()

In [44]:
print(len(train_dataset))
print(len(train_dataset[0])) # all three lists
print(len(train_dataset[0]["input_ids"]))
print(type(train_dataset))

4000
3
512
<class 'list'>


In [45]:
# input ids - dialogue => token ids
# attention mask
# labels - target => summary token

# 1 => EOS(End of sequence) following 0 -> padding

### Working with the Model

In [46]:
# NLP => generation task

model = T5ForConditionalGeneration.from_pretrained("t5-small")

Loading weights: 100%|████████████████████████████████████████████████████████████| 131/131 [00:00<00:00, 11085.74it/s]


In [49]:
import torch

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

model.to(device)

T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [50]:
# Training Arguments

training_args = TrainingArguments(
    output_dir = "./results",

    num_train_epochs=6,
    weight_decay=0.01,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    eval_strategy="epoch",
    save_strategy="epoch",

    warmup_steps=500
    # 0 => lr default
)

In [51]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

In [52]:
# Train the Model
trainer.train()

/opt/anaconda3/envs/python/lib/python3.11/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,3.633576,0.374045
2,0.396839,0.351404
3,0.373197,0.346500
4,0.362731,0.342840
5,0.355176,0.341544
6,0.352126,0.340994


Writing model shards: 100%|██████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 10.04it/s]
/opt/anaconda3/envs/python/lib/python3.11/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  8.85it/s]
/opt/anaconda3/envs/python/lib/python3.11/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 11.96it/s]
/opt/anaconda3/envs/python/lib/python3.11/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now

TrainOutput(global_step=3000, training_loss=0.9122742207845053, metrics={'train_runtime': 3077.2202, 'train_samples_per_second': 7.799, 'train_steps_per_second': 0.975, 'total_flos': 3248203235328000.0, 'train_loss': 0.9122742207845053, 'epoch': 6.0})

In [54]:
# we do model load => fine-tune => save the model
model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model")

Writing model shards: 100%|██████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  4.83it/s]


('./saved_summary_model/tokenizer_config.json',
 './saved_summary_model/tokenizer.json')

In [56]:
model = T5ForConditionalGeneration.from_pretrained("./saved_summary_model")
tokenizer = T5Tokenizer.from_pretrained("./saved_summary_model")

Loading weights: 100%|█████████████████████████████████████████████████████████████| 131/131 [00:00<00:00, 6779.20it/s]


### Testing the core logic for summarization

In [62]:
model = model.to(device)

def summarize_dialogue(dialogue):
    dialogue = clean_data(dialogue) # clean dialogue

    # tokenize
    inputs = tokenizer(
        dialogue,
        padding="max_length",
        max_length=512,
        truncation=True,
        return_tensors="pt"
    ).to(device)

    # generate the summary => token ids(tokens)
    targets = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=150,
        num_beams=4, # check 4 possible outputs => take best
        early_stopping=True
    )

    # token ids to text(summary) => decoding
    summary = tokenizer.decode(targets[0], skip_special_tokens=True) # EOS, SEP
    return summary

In [63]:
test_dialogue = """
Maya: Are we still meeting Friday to plan the launch?
Leo: Can we move it to Thursday? The design review is Friday morning.
Maya: Thursday works. How about 2 p.m.?
Leo: I have a call then. I’m free at 3:30.
Maya: Great. I’ll book the small conference room for Thursday at 3:30 and share the launch checklist beforehand.
"""

summary = summarize_dialogue(test_dialogue)
print(summary)

maya will book the small conference room for thursday at 3:30 and share the launch checklist beforehand.
